# Actividad 16 — Reentrenamiento con Dataset Extendido 2019-2025

Reentrena los 3 modelos clave con el dataset extendido (80 meses, 2019-01 a 2025-08) bajo condiciones identicas al experimento original.

| Modelo | Arquitectura | OUT_DIR |
|--------|-------------|---------|
| GE sin NLP | DualLSTM-BahdanauAttention (64/64) | `resultados/ge_ext/` |
| GM v3 con NLP | DualLSTM + PCA + Dropout NLP=0.5 | `resultados/gm_v3_ext/` |
| XGBoost | Grid search TimeSeriesSplit | `resultados/xgboost_ext/` |

**Hiperparametros fijos:** Adam lr=1e-3, batch=8, patience=15, seed=42, timesteps=6

In [1]:
import os, json, warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import tensorflow as tf
tf.get_logger().setLevel('ERROR')
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import xgboost as xgb
import matplotlib
matplotlib.use('Agg')

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

ROOT = Path.cwd()
while not (ROOT / 'CLAUDE.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA_PATH = ROOT / 'data/processed/master_escalado_extendido.csv'

TIMESTEPS     = 6
LSTM_UNITS    = 64
ATTN_UNITS    = 64
DROPOUT       = 0.30
L2_REG        = 0.001
LEARNING_RATE = 1e-3
EPOCHS        = 300
BATCH_SIZE    = 8
PATIENCE      = 15
N_TEST        = 12

print(f'TF: {tf.__version__}  |  XGB: {xgb.__version__}')
print(f'Data: {DATA_PATH.exists()}')

TF: 2.21.0  |  XGB: 3.2.0
Data: True


In [2]:
# Cargar dataset extendido y preparar features comunes
df_all = pd.read_csv(DATA_PATH, parse_dates=['fecha_evento'])
df_all = df_all.sort_values('fecha_evento').reset_index(drop=True)

TARGET = 'produccion_t'

# Crear lags de produccion_t (igual que GE original)
df_all['lag_1'] = df_all[TARGET].shift(1)
df_all['lag_3'] = df_all[TARGET].shift(3)
df_all['lag_6'] = df_all[TARGET].shift(6)
df_all = df_all.dropna().reset_index(drop=True)

n_total = len(df_all)
n_test  = N_TEST
n_train = n_total - n_test

print(f'Dataset con lags: {df_all.shape}')
print(f'Rango: {df_all["fecha_evento"].min().date()} -> {df_all["fecha_evento"].max().date()}')
print(f'n_train={n_train}  n_test={n_test}  n_total={n_total}')

# Features estructurales (canal B del GE) — sin NLP
STRUCT_COLS = [
    'lag_1', 'lag_3', 'lag_6',
    'precio_chacra_kg',
    'num_emergencias', 'total_afectados', 'hectareas_cultivo_perdidas',
    'ALLSKY_SFC_SW_DWN', 'PRECTOTCORR', 'QV2M', 'RH2M',
    'T2M', 'T2M_MAX', 'T2M_MIN', 'WS2M',
    'lat', 'lon',
    'month_sin', 'month_cos', 'mes_num',
    'trimestre_num', 'trimestre_sin', 'trimestre_cos',
]
NLP_COLS = ['nlp_index', 'nlp_index_lag1']

print(f'Struct features: {len(STRUCT_COLS)}')
print(f'NLP features: {len(NLP_COLS)}')
print(f'Columnas disponibles: {df_all.columns.tolist()}')

Dataset con lags: (74, 29)
Rango: 2019-07-01 -> 2025-08-01
n_train=62  n_test=12  n_total=74
Struct features: 23
NLP features: 2
Columnas disponibles: ['fecha_evento', 'produccion_t', 'precio_chacra_kg', 'num_emergencias', 'total_afectados', 'hectareas_cultivo_perdidas', 'ALLSKY_SFC_SW_DWN', 'PRECTOTCORR', 'QV2M', 'RH2M', 'T2M', 'T2M_MAX', 'T2M_MIN', 'WS2M', 'lat', 'lon', 'month_sin', 'month_cos', 'mes_num', 'trimestre_num', 'trimestre_sin', 'trimestre_cos', 'nlp_index', 'nlp_index_lag1', 'tiene_nlp', 'es_shock', 'lag_1', 'lag_3', 'lag_6']


In [3]:
# Funciones compartidas
class BahdanauAttention(keras.layers.Layer):
    def __init__(self, units, **kwargs):
        super().__init__(**kwargs)
        self.units    = units
        self.W_query  = layers.Dense(units, use_bias=False)
        self.W_values = layers.Dense(units, use_bias=False)
        self.V        = layers.Dense(1, use_bias=False)

    def call(self, query, values):
        q_exp  = tf.expand_dims(self.W_query(query), axis=1)
        energy = self.V(tf.nn.tanh(self.W_values(values) + q_exp))
        alpha  = tf.nn.softmax(energy, axis=1)
        ctx    = tf.reduce_sum(alpha * values, axis=1)
        return ctx, alpha

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'units': self.units})
        return cfg

def make_dual_sequences(target_arr, exog_arr, seq_len):
    Xa, Xb, Y = [], [], []
    for i in range(len(target_arr) - seq_len):
        Xa.append(target_arr[i:i+seq_len].reshape(-1, 1))
        Xb.append(exog_arr[i:i+seq_len])
        Y.append(target_arr[i+seq_len])
    return np.array(Xa, dtype=np.float32), np.array(Xb, dtype=np.float32), np.array(Y, dtype=np.float32)

def make_seq(Xs, Xn, y, ts):
    a, b, c = [], [], []
    for i in range(ts, len(Xs)):
        a.append(Xs[i-ts:i])
        b.append(Xn[i-ts:i])
        c.append(y[i])
    return np.array(a), np.array(b), np.array(c)

def compute_metrics(y_true, y_pred):
    mae  = float(mean_absolute_error(y_true, y_pred))
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    r2   = float(r2_score(y_true, y_pred))
    return {'MAE': mae, 'RMSE': rmse, 'R2': r2}

def compute_shock_metrics(y_true, y_pred, df_test_slice):
    variacion = df_test_slice[TARGET].pct_change().abs() * 100
    idx_shock = variacion[variacion > 20].index.tolist()
    if len(idx_shock) == 0:
        return 0, float('nan'), float('nan')
    local_idx = [df_test_slice.index.get_loc(i) for i in idx_shock]
    mae_shock  = float(mean_absolute_error(y_true[local_idx], y_pred[local_idx]))
    mae_normal_idx = [j for j in range(len(y_true)) if j not in local_idx]
    mae_normal = float(mean_absolute_error(y_true[mae_normal_idx], y_pred[mae_normal_idx])) if mae_normal_idx else float('nan')
    return len(idx_shock), mae_shock, mae_normal

def save_results(out_dir, model_name, metrics, fechas, y_true, y_pred):
    out_dir.mkdir(parents=True, exist_ok=True)
    metrics['modelo'] = model_name
    with open(out_dir / 'metricas.json', 'w') as f:
        json.dump(metrics, f, indent=2)
    pd.DataFrame({
        'fecha': fechas, 'real': y_true, 'predicho': y_pred
    }).to_csv(out_dir / 'predicciones.csv', index=False)
    print(f'  Guardado en {out_dir}/')

print('Funciones compartidas definidas.')

Funciones compartidas definidas.


---
## MODELO 1 — GE sin NLP (DualLSTM-BahdanauAttention)

Arquitectura identica a `actividad_14_ge_lstm_attention.ipynb`:
- Canal A: produccion_t (1 feature)
- Canal B: 23 features estructurales (lags + precio + INDECI + NASA + geo + temporal)
- LSTM(64) + BahdanauAttention(64) por canal
- Dense(64,relu) -> Dropout(0.15) -> Dense(16,relu) -> Dense(1)
- Prediccion multi-step recursiva

In [4]:
# ======================================================================
# MODELO 1: GE sin NLP
# ======================================================================
tf.random.set_seed(SEED); np.random.seed(SEED)

OUT_GE = ROOT / 'resultados/ge_ext'

# Split
df_train_ge = df_all.iloc[:n_train].copy()
df_test_ge  = df_all.iloc[n_train:].copy()

y_train_ge = df_train_ge[TARGET].values
y_test_ge  = df_test_ge[TARGET].values
B_train_ge = df_train_ge[STRUCT_COLS].values
B_test_ge  = df_test_ge[STRUCT_COLS].values

# Scalers (fit solo en train, como en el original)
scaler_a_ge = StandardScaler()
y_train_sc_ge = scaler_a_ge.fit_transform(y_train_ge.reshape(-1,1)).flatten()
y_test_sc_ge  = scaler_a_ge.transform(y_test_ge.reshape(-1,1)).flatten()

scaler_b_ge = StandardScaler()
B_train_sc_ge = scaler_b_ge.fit_transform(B_train_ge).astype(np.float32)
B_test_sc_ge  = scaler_b_ge.transform(B_test_ge).astype(np.float32)

idx_lag1 = STRUCT_COLS.index('lag_1')
idx_lag3 = STRUCT_COLS.index('lag_3')
idx_lag6 = STRUCT_COLS.index('lag_6')

# Secuencias para entrenamiento
Xa_train_ge, Xb_train_ge, y_seq_ge = make_dual_sequences(y_train_sc_ge, B_train_sc_ge, TIMESTEPS)
n_seq = len(Xa_train_ge)
n_val = max(3, int(n_seq * 0.15))
n_tr  = n_seq - n_val

Xa_tr, Xa_val = Xa_train_ge[:n_tr], Xa_train_ge[n_tr:]
Xb_tr, Xb_val = Xb_train_ge[:n_tr], Xb_train_ge[n_tr:]
y_tr, y_val   = y_seq_ge[:n_tr], y_seq_ge[n_tr:]

print(f'GE: Train={n_train} Test={n_test}')
print(f'Secuencias: train={n_tr} val={n_val}')

# Build model (identico a actividad_14)
reg = regularizers.l2(L2_REG)
inp_a = keras.Input(shape=(TIMESTEPS, 1), name='canal_a')
h_a = layers.LSTM(LSTM_UNITS, return_sequences=True, kernel_regularizer=reg, recurrent_regularizer=reg, name='lstm_a')(inp_a)
h_a = layers.Dropout(DROPOUT, name='drop_lstm_a')(h_a)
q_a = h_a[:, -1, :]
ctx_a, _ = BahdanauAttention(ATTN_UNITS, name='attn_a')(q_a, h_a)

inp_b = keras.Input(shape=(TIMESTEPS, len(STRUCT_COLS)), name='canal_b')
h_b = layers.LSTM(LSTM_UNITS, return_sequences=True, kernel_regularizer=reg, recurrent_regularizer=reg, name='lstm_b')(inp_b)
h_b = layers.Dropout(DROPOUT, name='drop_lstm_b')(h_b)
q_b = h_b[:, -1, :]
ctx_b, _ = BahdanauAttention(ATTN_UNITS, name='attn_b')(q_b, h_b)

merged = layers.Concatenate(name='fusion')([ctx_a, ctx_b])
x = layers.Dense(64, activation='relu', kernel_regularizer=reg, name='dense_1')(merged)
x = layers.Dropout(DROPOUT/2, name='drop_head')(x)
x = layers.Dense(16, activation='relu', name='dense_2')(x)
output = layers.Dense(1, name='output')(x)

model_ge = keras.Model(inputs=[inp_a, inp_b], outputs=output, name='GE_DualLSTM_ext')
model_ge.compile(optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE), loss='mse', metrics=['mae'])

print(f'Parametros: {model_ge.count_params():,}')

# Train
history_ge = model_ge.fit(
    [Xa_tr, Xb_tr], y_tr,
    validation_data=([Xa_val, Xb_val], y_val),
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    callbacks=[
        EarlyStopping(monitor='val_loss', patience=PATIENCE, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=8, min_lr=1e-6, verbose=0),
    ],
    verbose=0, shuffle=False,
)
print(f'Epocas: {len(history_ge.history["loss"])}  Best val_loss: {min(history_ge.history["val_loss"]):.6f}')

# Prediccion multi-step recursiva (identica a actividad_14)
canal_a_window = y_train_sc_ge[-TIMESTEPS:].copy()
pred_buffer_sc = list(y_train_sc_ge[-6:])
preds_sc_ge = []

for step in range(n_test):
    x_a = canal_a_window.reshape(1, TIMESTEPS, 1).astype(np.float32)
    start_b = max(0, step - TIMESTEPS + 1)
    b_avail = B_test_sc_ge[start_b:step+1]
    if len(b_avail) < TIMESTEPS:
        pad_b = np.tile(B_train_sc_ge[-1], (TIMESTEPS - len(b_avail), 1))
        b_window = np.vstack([pad_b, b_avail]).copy()
    else:
        b_window = b_avail.copy()
    for t in range(TIMESTEPS):
        def get_pred(off, buf=pred_buffer_sc):
            pos = len(buf) - off
            return buf[pos] if pos >= 0 else 0.0
        b_window[t, idx_lag1] = get_pred(1)
        b_window[t, idx_lag3] = get_pred(3)
        b_window[t, idx_lag6] = get_pred(6)
    x_b = b_window.reshape(1, TIMESTEPS, len(STRUCT_COLS)).astype(np.float32)
    pred_sc = float(model_ge.predict([x_a, x_b], verbose=0)[0, 0])
    preds_sc_ge.append(pred_sc)
    pred_buffer_sc.append(pred_sc)
    canal_a_window = np.roll(canal_a_window, -1)
    canal_a_window[-1] = pred_sc

y_pred_ge = scaler_a_ge.inverse_transform(np.array(preds_sc_ge).reshape(-1,1)).flatten()
y_true_ge = y_test_ge

metrics_ge = compute_metrics(y_true_ge, y_pred_ge)
n_shocks_ge, mae_shock_ge, mae_normal_ge = compute_shock_metrics(y_true_ge, y_pred_ge, df_test_ge)
metrics_ge['n_shocks'] = n_shocks_ge
metrics_ge['MAE_shock'] = mae_shock_ge
metrics_ge['MAE_normal'] = mae_normal_ge
if not np.isnan(mae_shock_ge):
    metrics_ge['deterioro_shock_pct'] = (mae_shock_ge - metrics_ge['MAE']) / metrics_ge['MAE'] * 100

fechas_ge = df_test_ge['fecha_evento'].dt.strftime('%Y-%m-%d').values
save_results(OUT_GE, 'GE_DualLSTM_ext', metrics_ge, fechas_ge, y_true_ge, y_pred_ge)

print(f'\n  GE ext: MAE={metrics_ge["MAE"]:.4f}  RMSE={metrics_ge["RMSE"]:.4f}  R2={metrics_ge["R2"]:.4f}')
print(f'  Shocks: {n_shocks_ge}/{n_test}  MAE_shock={mae_shock_ge:.4f}  MAE_normal={mae_normal_ge:.4f}')

GE: Train=62 Test=12
Secuencias: train=48 val=8


Parametros: 65,249


Restoring model weights from the end of the best epoch: 300.


Epocas: 300  Best val_loss: 0.028183


  Guardado en C:\Machine-learming\Machine-Learning-Multimodal--Agro-NLP-Clima-\resultados\ge_ext/

  GE ext: MAE=0.0035  RMSE=0.0048  R2=-402.6776
  Shocks: 0/12  MAE_shock=nan  MAE_normal=nan


---
## MODELO 2 — GM v3 con NLP (DualLSTM + PCA + Dropout NLP)

Arquitectura identica a `actividad_15v5_gm_nlp_v2.ipynb`:
- Canal A (struct): PCA(0.95) de features estructurales -> LSTM(64) + Softmax Attention
- Canal B (NLP): nlp_index + nlp_index_lag1 -> LSTM(16) + Dropout(0.5)
- tiene_nlp como feature adicional en canal A
- Dense(32,relu,L2=1e-4) -> Dropout(0.2) -> Dense(16,relu) -> Dense(1)
- Prediccion directa (no recursiva)

In [5]:
# ======================================================================
# MODELO 2: GM v3 con NLP
# ======================================================================
tf.random.set_seed(SEED); np.random.seed(SEED)

OUT_GM = ROOT / 'resultados/gm_v3_ext'

# Features para GM v3: struct (sin lags) + tiene_nlp
GM_STRUCT = [c for c in df_all.columns if c not in
             ['fecha_evento', TARGET, 'nlp_index', 'nlp_index_lag1',
              'tiene_nlp', 'es_shock', 'lag_1', 'lag_3', 'lag_6']]
GM_STRUCT = GM_STRUCT + ['tiene_nlp']

df_train_gm = df_all.iloc[:n_train].copy()
df_test_gm  = df_all.iloc[n_train:].copy()

# Scalers
scaler_s_gm = StandardScaler()
scaler_n_gm = StandardScaler()
scaler_y_gm = StandardScaler()

Xs_tr_raw = scaler_s_gm.fit_transform(df_train_gm[GM_STRUCT])
Xs_te_raw = scaler_s_gm.transform(df_test_gm[GM_STRUCT])
Xn_tr_raw = scaler_n_gm.fit_transform(df_train_gm[NLP_COLS])
Xn_te_raw = scaler_n_gm.transform(df_test_gm[NLP_COLS])
y_tr_sc_gm = scaler_y_gm.fit_transform(df_train_gm[[TARGET]])
y_te_sc_gm = scaler_y_gm.transform(df_test_gm[[TARGET]])

# PCA M4 (0.95 varianza explicada)
pca = PCA(n_components=0.95, random_state=SEED)
Xs_tr_pca = pca.fit_transform(Xs_tr_raw)
Xs_te_pca = pca.transform(Xs_te_raw)
n_comp = pca.n_components_
print(f'PCA: {Xs_tr_raw.shape[1]} -> {n_comp} componentes')

# Secuencias
Xs_seq_tr, Xn_seq_tr, y_seq_tr_gm = make_seq(Xs_tr_pca, Xn_tr_raw, y_tr_sc_gm, TIMESTEPS)
Xs_seq_te, Xn_seq_te, y_seq_te_gm = make_seq(Xs_te_pca, Xn_te_raw, y_te_sc_gm, TIMESTEPS)

print(f'GM v3: Xs_train={Xs_seq_tr.shape} Xn_train={Xn_seq_tr.shape}')
print(f'       Xs_test={Xs_seq_te.shape}  Xn_test={Xn_seq_te.shape}')

# Build model (identico a actividad_15v5)
def build_gm_v3(ss, ns, units=64, drs=0.2, drn=0.5):
    inp_s = layers.Input(shape=ss, name='inp_struct')
    h  = layers.LSTM(units, return_sequences=True)(inp_s)
    sc = layers.Dense(1, activation='tanh')(h)
    sw = layers.Softmax(axis=1)(sc)
    ca = layers.Multiply()([h, sw])
    ca = layers.Lambda(lambda x: tf.reduce_sum(x, axis=1))(ca)
    ca = layers.Dropout(drs)(ca)
    inp_n = layers.Input(shape=ns, name='inp_nlp')
    cb = layers.LSTM(16, return_sequences=False)(inp_n)
    cb = layers.Dropout(drn, name='dropout_nlp_M3')(cb)
    mg = layers.Concatenate()([ca, cb])
    x  = layers.Dense(32, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(mg)
    x  = layers.Dropout(0.2)(x)
    x  = layers.Dense(16, activation='relu')(x)
    out = layers.Dense(1)(x)
    return keras.Model(inputs=[inp_s, inp_n], outputs=out, name='GM_v3_ext')

ss = (Xs_seq_tr.shape[1], Xs_seq_tr.shape[2])
ns = (Xn_seq_tr.shape[1], Xn_seq_tr.shape[2])
model_gm = build_gm_v3(ss, ns)
model_gm.compile(optimizer=keras.optimizers.Adam(LEARNING_RATE), loss='mse', metrics=['mae'])

print(f'Parametros: {model_gm.count_params():,}')

# Train
history_gm = model_gm.fit(
    [Xs_seq_tr, Xn_seq_tr], y_seq_tr_gm,
    epochs=200, batch_size=BATCH_SIZE, validation_split=0.2,
    callbacks=[
        EarlyStopping(monitor='val_loss', patience=PATIENCE, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, min_lr=1e-6, verbose=0),
    ],
    shuffle=False, verbose=0,
)
print(f'Epocas: {len(history_gm.history["loss"])}  Best val_loss: {min(history_gm.history["val_loss"]):.6f}')

# Prediccion directa
y_pred_sc_gm = model_gm.predict([Xs_seq_te, Xn_seq_te], verbose=0)
y_pred_gm = scaler_y_gm.inverse_transform(y_pred_sc_gm).flatten()
y_true_gm = scaler_y_gm.inverse_transform(y_seq_te_gm).flatten()

metrics_gm = compute_metrics(y_true_gm, y_pred_gm)
# Shock analysis on the test portion aligned with sequences
df_test_seq = df_test_gm.iloc[TIMESTEPS:].copy()
n_shocks_gm, mae_shock_gm, mae_normal_gm = compute_shock_metrics(y_true_gm, y_pred_gm, df_test_seq)
metrics_gm['n_shocks'] = n_shocks_gm
metrics_gm['MAE_shock'] = mae_shock_gm
metrics_gm['MAE_normal'] = mae_normal_gm
if not np.isnan(mae_shock_gm):
    metrics_gm['deterioro_shock_pct'] = (mae_shock_gm - metrics_gm['MAE']) / metrics_gm['MAE'] * 100

fechas_gm = df_test_seq['fecha_evento'].dt.strftime('%Y-%m-%d').values
save_results(OUT_GM, 'GM_v3_NLP_ext', metrics_gm, fechas_gm, y_true_gm, y_pred_gm)

print(f'\n  GM v3 ext: MAE={metrics_gm["MAE"]:.4f}  RMSE={metrics_gm["RMSE"]:.4f}  R2={metrics_gm["R2"]:.4f}')
print(f'  Shocks: {n_shocks_gm}/{len(y_true_gm)}  MAE_shock={mae_shock_gm:.4f}  MAE_normal={mae_normal_gm:.4f}')

PCA: 21 -> 6 componentes
GM v3: Xs_train=(56, 6, 6) Xn_train=(56, 6, 2)
       Xs_test=(6, 6, 6)  Xn_test=(6, 6, 2)
Parametros: 22,594


Epoch 25: early stopping


Restoring model weights from the end of the best epoch: 10.


Epocas: 25  Best val_loss: 0.007292
  Guardado en C:\Machine-learming\Machine-Learning-Multimodal--Agro-NLP-Clima-\resultados\gm_v3_ext/

  GM v3 ext: MAE=0.0505  RMSE=0.0632  R2=-200910.0077
  Shocks: 0/6  MAE_shock=nan  MAE_normal=nan


---
## MODELO 3 — XGBoost

Mismo pipeline que `actividad_15v3_xgboost_competidor.ipynb`:
- Features: estructurales + NLP + lag (1,2,3,6) + rolling stats
- Grid search con TimeSeriesSplit(n_splits=3)
- Prediccion directa

In [6]:
# ======================================================================
# MODELO 3: XGBoost
# ======================================================================
np.random.seed(SEED)

OUT_XGB = ROOT / 'resultados/xgboost_ext'

# Preparar features XGBoost (lags adicionales + rolling)
df_xgb = df_all.copy()
df_xgb['prod_lag2'] = df_xgb[TARGET].shift(2)
df_xgb['prod_roll3_mean'] = df_xgb[TARGET].shift(1).rolling(3).mean()
df_xgb['prod_roll6_mean'] = df_xgb[TARGET].shift(1).rolling(6).mean()
df_xgb['prod_roll3_std']  = df_xgb[TARGET].shift(1).rolling(3).std()
df_xgb = df_xgb.dropna().reset_index(drop=True)

# Features: todo excepto fecha, target, es_shock
XGB_FEATURES = [c for c in df_xgb.columns
                if c not in ['fecha_evento', TARGET, 'es_shock']]

n_total_xgb = len(df_xgb)
n_test_xgb  = N_TEST
n_train_xgb = n_total_xgb - n_test_xgb

df_train_xgb = df_xgb.iloc[:n_train_xgb]
df_test_xgb  = df_xgb.iloc[n_train_xgb:]

X_train_xgb = df_train_xgb[XGB_FEATURES].values
y_train_xgb = df_train_xgb[TARGET].values
X_test_xgb  = df_test_xgb[XGB_FEATURES].values
y_test_xgb  = df_test_xgb[TARGET].values

print(f'XGBoost: Train={n_train_xgb} Test={n_test_xgb}')
print(f'Features: {len(XGB_FEATURES)}')

# Grid search con TimeSeriesSplit (identico al original)
from sklearn.model_selection import TimeSeriesSplit
from itertools import product

param_grid = {
    'max_depth':        [2, 3, 4],
    'n_estimators':     [50, 100, 200],
    'learning_rate':    [0.05, 0.1, 0.2],
    'subsample':        [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
}
tscv = TimeSeriesSplit(n_splits=3)
best_mae_cv = float('inf')
best_params = {}

combos = list(product(
    param_grid['max_depth'], param_grid['n_estimators'],
    param_grid['learning_rate'], param_grid['subsample'],
    param_grid['colsample_bytree']
))
print(f'Evaluando {len(combos)} combinaciones...')

for md, ne, lr, ss_p, cb in combos:
    maes = []
    for tr_idx, va_idx in tscv.split(X_train_xgb):
        m = xgb.XGBRegressor(max_depth=md, n_estimators=ne, learning_rate=lr,
                             subsample=ss_p, colsample_bytree=cb,
                             random_state=SEED, verbosity=0)
        m.fit(X_train_xgb[tr_idx], y_train_xgb[tr_idx])
        maes.append(mean_absolute_error(y_train_xgb[va_idx], m.predict(X_train_xgb[va_idx])))
    mae_cv = np.mean(maes)
    if mae_cv < best_mae_cv:
        best_mae_cv = mae_cv
        best_params = {'max_depth': md, 'n_estimators': ne, 'learning_rate': lr,
                       'subsample': ss_p, 'colsample_bytree': cb}

print(f'Mejor MAE CV: {best_mae_cv:.4f}')
print(f'Params: {best_params}')

# Entrenar modelo final
model_xgb = xgb.XGBRegressor(**best_params, random_state=SEED, verbosity=0)
model_xgb.fit(X_train_xgb, y_train_xgb)
y_pred_xgb = model_xgb.predict(X_test_xgb)
y_true_xgb = y_test_xgb

metrics_xgb = compute_metrics(y_true_xgb, y_pred_xgb)
n_shocks_xgb, mae_shock_xgb, mae_normal_xgb = compute_shock_metrics(y_true_xgb, y_pred_xgb, df_test_xgb)
metrics_xgb['n_shocks'] = n_shocks_xgb
metrics_xgb['MAE_shock'] = mae_shock_xgb
metrics_xgb['MAE_normal'] = mae_normal_xgb
metrics_xgb['best_params'] = best_params
if not np.isnan(mae_shock_xgb):
    metrics_xgb['deterioro_shock_pct'] = (mae_shock_xgb - metrics_xgb['MAE']) / metrics_xgb['MAE'] * 100

fechas_xgb = df_test_xgb['fecha_evento'].dt.strftime('%Y-%m-%d').values
save_results(OUT_XGB, 'XGBoost_ext', metrics_xgb, fechas_xgb, y_true_xgb, y_pred_xgb)

print(f'\n  XGBoost ext: MAE={metrics_xgb["MAE"]:.4f}  RMSE={metrics_xgb["RMSE"]:.4f}  R2={metrics_xgb["R2"]:.4f}')
print(f'  Shocks: {n_shocks_xgb}/{n_test_xgb}  MAE_shock={mae_shock_xgb:.4f}  MAE_normal={mae_normal_xgb:.4f}')

XGBoost: Train=56 Test=12
Features: 30
Evaluando 108 combinaciones...


Mejor MAE CV: 0.0888
Params: {'max_depth': 4, 'n_estimators': 100, 'learning_rate': 0.2, 'subsample': 1.0, 'colsample_bytree': 0.8}
  Guardado en C:\Machine-learming\Machine-Learning-Multimodal--Agro-NLP-Clima-\resultados\xgboost_ext/

  XGBoost ext: MAE=0.0004  RMSE=0.0005  R2=-3.1564
  Shocks: 0/12  MAE_shock=nan  MAE_normal=nan


---
## Tabla Comparativa Final

MAE global y deterioro en shocks: dataset original (2021-2025) vs extendido (2019-2025)

In [7]:
# ======================================================================
# TABLA COMPARATIVA FINAL
# ======================================================================

# Metricas originales (del CLAUDE.md y notebooks ejecutados)
orig = {
    'GE sin NLP':  {'MAE': 0.0673, 'det_orig': '+2.3%'},
    'GM v3 NLP':   {'MAE': 0.0645, 'det_orig': '+11.7%'},
    'XGBoost':     {'MAE': 0.0471, 'det_orig': '+16.5%'},
}

# Metricas extendidas
ext = {
    'GE sin NLP':  metrics_ge,
    'GM v3 NLP':   metrics_gm,
    'XGBoost':     metrics_xgb,
}

print('=' * 90)
print('  COMPARATIVA: DATASET ORIGINAL (2021-2025) vs EXTENDIDO (2019-2025)')
print('=' * 90)
print(f"  {'Modelo':<16} {'MAE orig':>10} {'MAE ext':>10} {'RMSE ext':>10} {'R2 ext':>10} {'Det orig':>12} {'Det ext':>12}")
print('-' * 90)

for name in ['GE sin NLP', 'GM v3 NLP', 'XGBoost']:
    mae_o = orig[name]['MAE']
    mae_e = ext[name]['MAE']
    rmse_e = ext[name]['RMSE']
    r2_e  = ext[name]['R2']
    det_o = orig[name]['det_orig']

    det_e_val = ext[name].get('deterioro_shock_pct', float('nan'))
    det_e = f'{det_e_val:+.1f}%' if not np.isnan(det_e_val) else 'N/A'

    delta = (mae_e - mae_o) / mae_o * 100
    delta_str = f'({delta:+.1f}%)' if delta != 0 else ''

    print(f"  {name:<16} {mae_o:>10.4f} {mae_e:>10.4f} {rmse_e:>10.4f} {r2_e:>10.4f} {det_o:>12} {det_e:>12}  {delta_str}")

print('=' * 90)

# Resumen de shocks
print(f'\n  ANALISIS DE SHOCKS EN TEST (ultimos {N_TEST} meses)')
print(f"  {'Modelo':<16} {'Shocks':>8} {'MAE global':>10} {'MAE shock':>10} {'MAE normal':>10}")
print('-' * 60)
for name, m in [('GE sin NLP', metrics_ge), ('GM v3 NLP', metrics_gm), ('XGBoost', metrics_xgb)]:
    sh = m.get('n_shocks', 0)
    ms = m.get('MAE_shock', float('nan'))
    mn = m.get('MAE_normal', float('nan'))
    ms_s = f'{ms:.4f}' if not np.isnan(ms) else 'N/A'
    mn_s = f'{mn:.4f}' if not np.isnan(mn) else 'N/A'
    print(f"  {name:<16} {sh:>8} {m['MAE']:>10.4f} {ms_s:>10} {mn_s:>10}")
print('=' * 60)

# Guardar comparativa
comparativa = {
    'dataset': 'master_escalado_extendido.csv',
    'n_train': n_train, 'n_test': n_test,
    'modelos': {
        'GE_sin_NLP': {'orig': orig['GE sin NLP'], 'ext': metrics_ge},
        'GM_v3_NLP':  {'orig': orig['GM v3 NLP'],  'ext': metrics_gm},
        'XGBoost':    {'orig': orig['XGBoost'],     'ext': metrics_xgb},
    }
}
with open(ROOT / 'resultados/comparativa_extendido.json', 'w') as f:
    json.dump(comparativa, f, indent=2, default=str)
print(f'\nComparativa guardada en resultados/comparativa_extendido.json')

  COMPARATIVA: DATASET ORIGINAL (2021-2025) vs EXTENDIDO (2019-2025)
  Modelo             MAE orig    MAE ext   RMSE ext     R2 ext     Det orig      Det ext
------------------------------------------------------------------------------------------
  GE sin NLP           0.0673     0.0035     0.0048  -402.6776        +2.3%          N/A  (-94.8%)
  GM v3 NLP            0.0645     0.0505     0.0632 -200910.0077       +11.7%          N/A  (-21.7%)
  XGBoost              0.0471     0.0004     0.0005    -3.1564       +16.5%          N/A  (-99.1%)

  ANALISIS DE SHOCKS EN TEST (ultimos 12 meses)
  Modelo             Shocks MAE global  MAE shock MAE normal
------------------------------------------------------------
  GE sin NLP              0     0.0035        N/A        N/A
  GM v3 NLP               0     0.0505        N/A        N/A
  XGBoost                 0     0.0004        N/A        N/A

Comparativa guardada en resultados/comparativa_extendido.json
